# Sample Exam Questions — by Topic

## What's covered

- 29 sample exam-style questions, grouped under the notebook each topic was taught in
- Each entry: the **scenario**, the **answer** with code, the **why**, and the **trap** the wrong answers usually plant
- Use this as a self-check after working through notebooks 01–09

## How questions are ordered

Reading order matches the curriculum:

1. **Foundations & Execution Model** (`01`) — cores, parallelism, Spark Connect, when *not* to use Spark, **lazy evaluation & wide/narrow**
2. **DataFrames** (`03`) — null checks
3. **Reading & Writing** (`04`) — schema, formats, recursion, JDBC, save modes
4. **Transformations & Aggregations** (`05`) — unionByName, time zones, dedup, strings, ordering
5. **Spark SQL & UDFs** (`06`) — temp views
6. **Structured Streaming** (`07`) — output modes, windowed aggregations, triggers, checkpoint locations
7. **Performance Tuning** (`08`) — coalesce vs. repartition, broadcast joins, driver GC, **wide vs. narrow / shuffle operations**
8. **Pandas API on Spark** (`09`) — converting back to a Spark DataFrame

Run this notebook on the same `local[*]` setup used by `01-spark-foundations-execution-model.ipynb`. Code cells use the same five-customer DataFrame so you can paste them into either notebook.

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("SampleQuestions")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

# The same five-customer set the rest of the series uses.
data = [
    ("CUST0001", "Aarav Sharma",  "Mumbai",    "aarav@example.com",  780),
    ("CUST0002", "Priya Nair",    "Delhi",     "priya@example.com",  650),
    ("CUST0003", "Rohan Gupta",   "Bengaluru", "rohan@example.com",  720),
    ("CUST0004", "Anjali Mehta",  "Pune",      None,                 810),
    ("CUST0005", "Vikram Reddy",  "Hyderabad", "vikram@example.com", 590),
]
df = spark.createDataFrame(data, ["customer_id", "full_name", "city", "email", "credit_score"])
df.show(truncate=False)

# 01 · Foundations & Execution Model

## Q1 — n workers × m cores: how many tasks run at once?

**Scenario.** A cluster has `n` worker nodes and each worker has `m` cores. How many partitions / tasks run in parallel?

**Answer.** *`n × m`* — total cores across the cluster equals max concurrent tasks.

**Why.** A Spark task is the unit that runs on **one core**, processing **one partition**. Tasks per second is gated by total cores; the number of *partitions* is a separate, data-driven number that controls *how the work is sliced*, not how much runs at once.

**Trap.** Confusing **partition count** (data-driven, often 200 after a shuffle) with **parallelism** (cluster-driven, cores). 1000 partitions on a 4-core cluster still run 4 at a time.

## Q2 — Point an existing app at a local Spark Connect server

**Scenario.** A Connect server is running at `sc://localhost:15002`. You want the existing app to talk to it with **minimal code change**. Two ways?

**Answer.** Either:

1. Build with **`.remote(...)`**:

   ```python
    spark = SparkSession.builder.remote("sc://localhost:15002").getOrCreate()
   ```

2. Set the **`SPARK_REMOTE`** env var and let `getOrCreate()` pick it up:

   ```bash
    export SPARK_REMOTE="sc://localhost:15002"
   ```

**Why.** Both paths route the existing `SparkSession.builder...getOrCreate()` call through Connect without other changes.

**Trap.** Calling `.master("sc://...")`. `master` is for classic mode — `remote` is the Connect equivalent.

## Q3 — Tiny CSV, one engineer, one machine. Is plain pandas enough?

**Scenario.** A CSV is a few MB; the workload is a single script on a laptop.

**Answer.** **Yes** — plain pandas (`pd.read_csv(...)`) is the right tool.

**Why.** Spark earns its complexity past the point where single-machine tools stop working — typically several to tens of GB. Below that, JVM startup, planning, and serialisation make Spark slower than pandas *and* harder to reason about.

**Trap.** Reaching for Spark by default. The exam rewards picking pandas when the scenario is clearly small-and-local.

## Q4 — When does Spark actually read the CSV, and which step shuffles?

**Scenario.** A standard read → filter → aggregate → action chain.

```python
 df    = spark.read.csv(\"/path/to/transactions.csv\", header=True, schema=my_schema)
 hi    = df.filter(col(\"amount\") > 100)
 agg   = hi.groupBy(\"city\").count()
 agg.show()
```

**What is true about how this runs?**

Three foundational rules in one chain:

1. **The CSV read is lazy.** `spark.read.csv(...)` doesn't read a byte — it builds a logical plan node. No I/O happens until an action runs.
2. **`filter` is a narrow transformation.** Each input partition produces its output partition independently — no data crosses the network. Catalyst pushes the predicate down close to the scan so unmatched rows aren't fully materialised.
3. **`groupBy(...).count()` is a wide transformation.** Rows for the same `city` may live in different partitions, so Spark **shuffles** by `city` before aggregating. The shuffle is a stage boundary.
4. **`.show()` is the action.** That's what triggers the whole DAG. None of the lines before it did any work.

**The execution shape:**

```text
 Stage 1                                   Stage 2 (after shuffle)
 ─────────────────────────                 ─────────────────────────
 read CSV partition  ──► filter ──► ─┐
 read CSV partition  ──► filter ──► ─┤── shuffle by city ──► count ──► show()
 read CSV partition  ──► filter ──► ─┘
```

**Why this matters.**

- Define the **whole** chain before the action. Catalyst optimises across the full chain — predicate pushdown into the scan, column pruning, join-strategy selection. Calling `.show()` after every step pays the full execution cost every time.
- The first action — `show()`, `count()`, `collect()`, `write` — is the only call that actually reads bytes off disk. Use **`.explain()`** to inspect the plan without triggering execution.
- The plan will show one `Exchange` operator (the shuffle for `groupBy`). The `filter` does not appear as a separate exchange — it's fused into the scan stage.

**Traps:**

- *\"`spark.read.csv` is what makes this slow.\"* No — the read is free; the action is what costs.
- *\"`filter` needs a shuffle to drop rows.\"* No — `filter` is per-partition; dropping rows doesn't move data.
- *\"Adding `.show()` between every step helps debugging.\"* Each `.show()` re-runs the chain from the source unless you `.cache()`. For debugging, prefer `.explain()`.
- *\"`groupBy` is narrow because it groups within each partition.\"* No — keys spread across partitions, so Spark must redistribute by key. That's a shuffle.

In [ ]:
from pyspark.sql.functions import col, count

# 1) Build the chain. NO I/O happens here.
high_score = df.filter(col("credit_score") > 700)
by_city    = high_score.groupBy("city").agg(count("*").alias("n"))
print("Chain defined. Nothing has executed yet.\n")

# 2) .explain() inspects the optimized plan WITHOUT triggering execution.
#    Look for one 'Exchange' node — that's the shuffle introduced by groupBy.
#    The 'PushedFilters' line under FileScan / InMemoryRelation shows the
#    filter predicate being pushed close to the read.
by_city.explain()

# 3) NOW .show() is the action that actually runs the whole DAG.
by_city.show()

# 03 · DataFrames

## Q5 — Filter rows where a column is null

**Scenario.** Find rows in `df` where `email` is missing.

**Answer.** `col("email").isNull()` — column-level null check, composes inside `filter` / `where`.

**Trap.** Writing `col("email") == None`. In Spark SQL semantics, `NULL == anything` is `NULL` (not `TRUE`), so the filter returns no rows. Use `isNull()` / `isNotNull()`.

In [ ]:
from pyspark.sql.functions import col

df.filter(col("email").isNull()).show(truncate=False)

# Inverse:
df.filter(col("email").isNotNull()).select("customer_id", "email").show(truncate=False)

# 04 · Reading & Writing

## Q6 — Parquet part-files have slightly different schemas. How do you reconcile?

**Scenario.** A folder of Parquet files where some files added new columns over time. You want Spark to read them as one DataFrame with the union of columns.

**Answer.** `.option("mergeSchema", "true")` on the reader.

```python
 spark.read.option("mergeSchema", "true").parquet("/path/to/dir")
```

**Why.** Spark reads the per-file footers and unions their schemas. Missing columns become `null`.

**Trap.** Spelling — it's `mergeSchema`, not `mergescheme`, `merge_schema`, or `parquet.mergeschema`. Also, this option is **expensive** on huge file counts because every footer must be read.

## Q7 — Read every file under a directory tree, regardless of subfolder layout

**Scenario.** Files live under arbitrary nested paths (`/root/a/b/c/file.csv`, `/root/x/y/file.csv`) and you want them all.

**Answer.** `.option("recursiveFileLookup", "true")`.

```python
 spark.read.option("recursiveFileLookup", "true").csv("/root")
```

**The nuance everyone misses.** `recursiveFileLookup` **disables partition discovery**. The folders `year=2024/month=01/...` won't become columns. Two patterns:

- **Want `year` / `month` / `day` as columns** → use Hive-style partitioned paths and *do not* set `recursiveFileLookup`; Spark infers partitions automatically.
- **Want every file from a messy tree** → set `recursiveFileLookup=true`; you lose partition columns.
- **Want both** (rare) → use `basePath` to anchor partition inference while still letting Spark discover sub-paths.

## Q8 — Read JSON with a predefined schema (skip inference)

**Scenario.** A nightly JSON load. Schema inference takes minutes; the schema is known and stable.

**Answer.** Pass the schema to the reader.

```python
 spark.read.schema(my_schema).json("/path/to/files")
```

**Why.** Inference scans the file (sometimes the whole file) to deduce types. Supplying a `StructType` skips that and **enforces** the types you specified — values that don't fit become `null` (or land in `_corrupt_record` if you wire that column).

**Trap.** Using `.option("inferSchema", "true")` for big files. That's what you're trying to avoid.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

my_schema = StructType([
    StructField("customer_id",  StringType(),  False),
    StructField("full_name",    StringType(),  True),
    StructField("credit_score", IntegerType(), True),
])
# Demo with an in-memory JSON RDD — the API is identical with .json(path).
sample = spark.sparkContext.parallelize([
    '{"customer_id":"CUST0001","full_name":"Aarav Sharma","credit_score":780}',
    '{"customer_id":"CUST0002","full_name":"Priya Nair","credit_score":650}',
])
spark.read.schema(my_schema).json(sample).show(truncate=False)

## Q9 — CSV with a header row and `;` separator

**Scenario.** European-format CSV: first row is column names, columns are separated by semicolons.

**Answer.**

```python
 spark.read \
      .option("header", "true") \
      .option("sep", ";") \
      .csv("/path/to/file.csv")
```

**Why.** `header = true` tells Spark to use row 1 as column names. `sep` (alias `delimiter`) overrides the default comma.

**Trap.** Hardcoding the delimiter in a regex or post-processing the rows. Reader options exist; use them.

## Q10 — `saveAsTable` should fail if the table already exists

**Scenario.** A daily job creates a brand-new table. If the table somehow already exists, the job must fail loudly rather than overwrite or append.

**Answer.** `.mode("errorIfExists")` — which is also the **default** when you don't specify a mode.

```python
 df.write.mode("errorIfExists").saveAsTable("silver.customers")
```

**The four save modes:**

| Mode | Behaviour |
|---|---|
| `errorIfExists` (default) | Throw if target exists |
| `append` | Add rows to existing target |
| `overwrite` | Replace existing target |
| `ignore` | Silently do nothing if target exists |

**Trap.** Forgetting that `errorIfExists` is the default — the question can phrase it as "use the default and fail loudly".

## Q11 — JDBC read: parallel splits and what each option does

**Scenario.** Reading a big Postgres table over JDBC; the read is single-threaded and slow.

**Answer.** Provide `partitionColumn`, `lowerBound`, `upperBound`, and `numPartitions` to parallelise the read across Spark executors.

```python
 (spark.read.format("jdbc")
     .option("url",       "jdbc:postgresql://...")
     .option("dbtable",   "public.orders")
     .option("user",      user)
     .option("password",  pw)
     .option("partitionColumn", "order_id")
     .option("lowerBound",      1)
     .option("upperBound",      10_000_000)
     .option("numPartitions",   8)
     .load())
```

**Why.** Spark issues `numPartitions` separate `SELECT ... WHERE partitionColumn BETWEEN ?` queries, each on a different range. Eight queries in parallel ≫ one. **Lowering** `numPartitions` reduces concurrent DB connections — useful if you're throttled.

**Trap.** Setting `partitionColumn` to something that isn't numeric / date and isn't well-distributed; bounds become meaningless and one partition gets all the rows.

## Q12 — `spark.sql(...).write.save(...)` — getting the call signature right

**Scenario.** You want to run a SQL query and write the result to a path in a non-default format.

**Answer.** Two correct shapes — pick whichever reads better:

```python
 spark.sql("SELECT * FROM customers") \
      .write \
      .mode("overwrite") \
      .save("/tmp/out", format="parquet")

 # or the chained-options form
 spark.sql("SELECT * FROM customers") \
      .write \
      .format("parquet") \
      .mode("overwrite") \
      .save("/tmp/out")
```

**Trap.** `save(path, mode)` — wrong. The **second positional arg of `save()` is the format**, not the mode. Mode goes through `.mode(...)`.

## Q13 — What does `mode("overwrite")` actually do?

**Scenario.** Rebuilding silver from bronze; the write target already has data from yesterday.

**Answer.** Conceptually: **replace** the data at the target.

- **File output**: delete existing files at the path, then write fresh ones.
- **Table output**: replace the table's data; schema can be replaced too with `option("overwriteSchema", "true")` on Delta.
- **Dynamic partition overwrite** (`spark.sql.sources.partitionOverwriteMode = dynamic`): only replace the partitions present in the incoming DataFrame; other partitions untouched.

**Trap.** Assuming `overwrite` always replaces *the whole table*. With dynamic partition overwrite enabled, only the affected partitions move.

# 05 · Data Transformations & Aggregations

## Q14 — Union two DataFrames where columns differ

**Scenario.** Two DataFrames carry mostly the same columns; one has an extra column the other doesn't.

**Answer.** `unionByName(df2, allowMissingColumns=True)` — matches by **name**, fills missing columns with `null`.

```python
 df1.unionByName(df2, allowMissingColumns=True)
```

**Why.** `union` (and the deprecated `unionAll`) match columns **by position**, so a mismatched schema silently shifts values into the wrong columns. `unionByName` matches by name. The `allowMissingColumns=True` flag tolerates the schema being a superset on one side.

**Trap.** Using `union` on schemas that drift over time.

In [ ]:
from pyspark.sql import Row
df_a = spark.createDataFrame([Row(customer_id="C1", credit_score=700)])
df_b = spark.createDataFrame([Row(customer_id="C2", credit_score=720, city="Pune")])

df_a.unionByName(df_b, allowMissingColumns=True).show(truncate=False)

## Q15 — Convert a UTC timestamp to local time

**Scenario.** Stored timestamps are UTC; you want them displayed in `America/New_York`.

**Answer.** `from_utc_timestamp(col("ts"), "America/New_York")`.

```python
 from pyspark.sql.functions import from_utc_timestamp
 df.withColumn("ts_ny", from_utc_timestamp("ts", "America/New_York"))
```

**Trap.** Using a region/state name like `"New Jersey"` — there is no such tz. Pick a city from the IANA tz database (`America/New_York`, `Asia/Kolkata`, `Europe/London`, …).

## Q16 — Remove duplicate rows

**Scenario.** A bronze load occasionally double-emits a row. You want one row per natural key.

**Answer.** `dropDuplicates(subset=[...])` — drop rows where the listed columns are duplicates.

```python
 df.dropDuplicates(["customer_id"])
```

**Why.** `distinct()` drops rows that are duplicates across *all* columns. `dropDuplicates(subset=...)` lets you scope to the key.

**Trap.** Assuming a particular row survives. With `dropDuplicates`, the survivor is **non-deterministic**. For "keep latest", use a window + `row_number()` over an audit timestamp.

In [ ]:
from pyspark.sql import Row
dups = spark.createDataFrame([
    Row(customer_id="CUST0001", v="a"),
    Row(customer_id="CUST0001", v="b"),
    Row(customer_id="CUST0002", v="a"),
])
dups.dropDuplicates(["customer_id"]).show()

## Q17 — Extract the local part of an email (`aarav@example.com` → `aarav`)

**Answer.** Two common approaches — exam can show either:

```python
 from pyspark.sql.functions import split, substring_index, col

 # split + index
 df.withColumn("local", split(col("email"), "@")[0])

 # substring_index — element-of-string before / after a delimiter
 df.withColumn("local", substring_index(col("email"), "@", 1))
```

`regexp_extract(col("email"), r"^([^@]+)@", 1)` is a third valid answer when the pattern is more complex.

**Trap.** Trying to do this with `lpad` / `rpad` or with Python string ops outside Spark (forces a `collect`).

In [ ]:
from pyspark.sql.functions import split, substring_index, col
(df.filter(col("email").isNotNull())
   .withColumn("local_split", split(col("email"), "@")[0])
   .withColumn("local_substridx", substring_index(col("email"), "@", 1))
   .select("email", "local_split", "local_substridx")
   .show(truncate=False))

## Q18 — `orderBy(desc("credit_score")).take(3)` — top 3 or bottom 3?

**Scenario.** A developer writes `df.orderBy(desc("credit_score")).take(3)` and reports the *3 customers with the lowest credit scores*.

**Answer.** That call returns the **top 3** (highest), not the bottom 3.

```python
 from pyspark.sql.functions import desc

 df.orderBy("credit_score").take(3)         # 3 LOWEST  (ascending default)
 df.orderBy(desc("credit_score")).take(3)   # 3 HIGHEST
```

**Trap.** Mistaking `desc` for "descending = smaller first". Descending means *largest first*, so `take(3)` returns the three largest.

## Q19 — Does `orderBy("age").take(5)` give the 5 OLDEST people over 30?

**Scenario.** A developer wants the **top 5 oldest people aged over 30** from `people`. They write:

```python
 (people
   .select("name", "age")
   .filter("age > 30")
   .orderBy("age")
   .take(5))
```

Does this return the answer they wanted?

**Answer.** **No.** It returns the **5 youngest** people over 30 — typically ages 31, 31, 32, 33, 34. The opposite of what they wanted.

**Why.** `orderBy("age")` is **ascending by default**. `take(5)` returns the **first 5 rows in the current order**. Ascending + take(5) = the 5 smallest values, so you get the youngest five who are still over 30.

**The fix — sort descending:**

```python
 from pyspark.sql.functions import col, desc

 (people
   .select("name", "age")
   .filter("age > 30")
   .orderBy(col("age").desc())     # or orderBy(desc("age"))
   .take(5))
```

**The full direction cheat sheet:**

| Expression | Direction | `.take(5)` returns |
|---|---|---|
| `orderBy("age")` | ascending (default) | 5 **youngest** |
| `orderBy(col("age").asc())` | ascending (explicit) | 5 **youngest** |
| `orderBy(col("age").desc())` | descending | 5 **oldest** |
| `orderBy(desc("age"))` | descending | 5 **oldest** |
| `orderBy(col("age").asc_nulls_last())` | ascending, nulls last | 5 youngest, nulls pushed to end |

**Traps:**

- *"`orderBy` figures out which direction I want."* No — default is **ascending**. You must explicitly say `desc` for largest-first.
- *"`take(N)` always returns the top N."* `take(N)` returns the **first N** rows in the current ordering; "top" depends on the sort.
- *"`orderBy` is narrow because it only sorts within partitions."* No — `orderBy` is **wide**. Spark needs every row from every partition to agree on the global order, which requires a **range-partitioned shuffle**. Look for `Exchange rangepartitioning` in the plan.
- *"`.tail(5)` would have given the oldest."* `.tail(5)` exists on Spark 3.0+ and returns the last 5 of an ordered DataFrame — but `desc` + `take` is the idiomatic answer the exam expects.

**Related:** Q18 covers the mirror trap — `orderBy(desc(...)).take(3)` returning the *top* 3, not the bottom 3. Both questions hit the same root: `orderBy` defaults to ascending, and `take(N)` returns the *first* N rows in that order.

In [ ]:
from pyspark.sql.functions import col, desc

# Bug: orderBy is ASCENDING by default — take(5) returns the LOWEST scores above 600.
wrong = (df.select("full_name", "credit_score")
           .filter(col("credit_score") > 600)
           .orderBy("credit_score")
           .take(5))

# Fix: orderBy DESCENDING — take(5) now returns the highest.
right = (df.select("full_name", "credit_score")
            .filter(col("credit_score") > 600)
            .orderBy(col("credit_score").desc())
            .take(5))

print("WRONG (asc, smallest first — what the buggy code returns):")
for r in wrong: print(" ", r.full_name, r.credit_score)

print("\nRIGHT (desc, largest first — what the user actually wanted):")
for r in right: print(" ", r.full_name, r.credit_score)

In [ ]:
from pyspark.sql.functions import desc
print("3 LOWEST:")
for r in df.orderBy("credit_score").take(3):
    print("  ", r.customer_id, r.credit_score)
print("3 HIGHEST:")
for r in df.orderBy(desc("credit_score")).take(3):
    print("  ", r.customer_id, r.credit_score)

# 06 · Spark SQL & UDFs

## Q20 — `createOrReplaceTempView` semantics

**Scenario.** Register a DataFrame as a SQL view. If a view with the same name already exists, what happens?

**Answer.** It is **silently replaced** — no error. Lifetime is the current `SparkSession`.

```python
 df.createOrReplaceTempView("customers")
 spark.sql("SELECT COUNT(*) FROM customers").show()
```

**Variants:**

- `createTempView(name)` — **fails** if a view with that name already exists.
- `createOrReplaceGlobalTempView(name)` — cross-session, prefixed with `global_temp.`.

**Trap.** Confusing `createOrReplaceTempView` with `saveAsTable`. The temp view is metadata-only; no data is written.

## Q21 — Complex SQL transformations from a DataFrame

**Scenario.** A multi-step aggregation and join is much cleaner expressed in SQL than in the DataFrame API.

**Answer.** Register a temp view and run `spark.sql(...)`.

```python
 df.createOrReplaceTempView("customers")
 spark.sql("""
     SELECT city,
            COUNT(*)              AS n,
            ROUND(AVG(credit_score),0) AS avg_score
     FROM   customers
     GROUP BY city
     ORDER BY avg_score DESC
 """).show()
```

**Why.** The same Catalyst plan runs regardless of whether you write SQL or DataFrame chain — but SQL is often clearer for complex transformations.

**Trap.** Re-writing the whole transformation in Python when SQL was already clearer.

In [ ]:
df.createOrReplaceTempView("customers")
spark.sql("""
    SELECT city,
           COUNT(*)                    AS num_customers,
           ROUND(AVG(credit_score), 0) AS avg_credit_score
    FROM customers
    GROUP BY city
    ORDER BY avg_credit_score DESC
""").show()

# 07 · Structured Streaming

## Q22 — Rewrite the entire result table every trigger

**Scenario.** A streaming aggregation; the sink should receive the **full updated result** each trigger, not just deltas.

**Answer.** `outputMode("complete")`.

```python
 (stream
    .groupBy("city").count()
    .writeStream
    .outputMode("complete")
    .format("console")
    .start())
```

**Why and constraint.** Complete mode rewrites the full result table every trigger. It is **only valid when the query contains an aggregation** — Spark needs to maintain state for it.

**The three output modes** the exam tests:

| Mode | Behaviour | Allowed when |
|---|---|---|
| `append` | Only emit *new* rows since last trigger | Query has no aggregation, OR has aggregation with a watermark |
| `update` | Emit rows that *changed* this trigger | Query has an aggregation |
| `complete` | Emit the full result table every trigger | Query has an aggregation |

**Trap.** Trying `complete` on a non-aggregating query — it errors at start.

## Q23 — Tumbling 10-minute window, trigger every 10 minutes

**Scenario.** Count events per 10-minute bucket; emit a new batch every 10 minutes.

**Answer.** Combine `window(...)` in the `groupBy` with `trigger(processingTime="10 minutes")` on the writer.

```python
 from pyspark.sql.functions import window, col

 (stream
    .groupBy(window(col("ts"), "10 minutes"))
    .count()
    .writeStream
    .trigger(processingTime="10 minutes")
    .outputMode("update")
    .start())
```

**Why.** Pass *one* duration to `window(...)` and you get a **tumbling** (non-overlapping) window. The trigger interval is independent — `processingTime="10 minutes"` fires a micro-batch every 10 minutes.

**Trap.** Confusing window duration with slide duration. `window(col("ts"), "10 minutes", "5 minutes")` would be a **sliding** window (10-min duration, sliding every 5 min).

## Q24 — `checkpointLocation`: resume after a restart instead of starting over

**Scenario.** The bank's card-transactions streaming job ran successfully for six hours, then the cluster died at 03:00. When the job restarts at 03:15, the team wants it to **pick up exactly where it left off** — no reprocessing the six hours, no missing the messages that landed during the outage.

**Answer.** Set **`checkpointLocation`** on the writer.

```python
 (stream
    .writeStream
    .option("checkpointLocation", "/Volumes/fintech/bronze/_ckpts/card_txns")
    .outputMode("append")
    .toTable("bronze.card_transactions_raw")
    .awaitTermination())
```

**What lives inside the checkpoint directory:**

| Subfolder | What it tracks |
|---|---|
| `offsets/` | Per-trigger snapshot of how far the source has been read (Kafka offsets, file paths consumed, etc.) |
| `commits/` | Per-trigger confirmation that the sink wrote successfully |
| `state/` | The state store for stateful operations (aggregations, stream-stream joins) |
| `metadata` | Streaming query identity — pins the query to this directory |
| `sources/` | Source-specific bookkeeping (Auto Loader's schema log lives nearby) |

**Why it works.** On restart, Spark reads the highest committed offset, replays only un-committed records, and re-runs the writer transactionally — that's **exactly-once semantics on the write path**. Sources that support replay (Kafka, Auto Loader, Delta) extend this to end-to-end exactly-once.

**Traps to avoid:**

- **No `checkpointLocation`** → restart re-processes from scratch (or fails to start, on Delta sinks).
- **Local filesystem path on a real cluster** → fine for `local[*]`, fatal on multi-node clusters (each node has its own disk). Use a UC volume, S3, ADLS, or GCS path.
- **Shared between two queries** → state corruption. Each streaming query needs its **own** dedicated checkpoint directory.
- **Deleting the directory between deploys** → the new run starts from offset 0. Treat checkpoints as part of the *data*, not the code.

In [ ]:
# Demo with the rate source — Run 1 emits some rows, then we stop.
# On Run 2 with the SAME checkpoint, Spark resumes from the next offset.
import shutil, time
from pathlib import Path

ckpt = "/tmp/spark_ckpt_demo"
out  = "/tmp/spark_ckpt_out"
shutil.rmtree(ckpt, ignore_errors=True)
shutil.rmtree(out,  ignore_errors=True)

def run_for(seconds):
    q = (spark.readStream.format("rate").option("rowsPerSecond", 1).load()
             .writeStream
             .option("checkpointLocation", ckpt)
             .format("parquet").option("path", out)
             .outputMode("append")
             .start())
    time.sleep(seconds)
    q.stop()

# Run 1 — generate a handful of rows.
run_for(6)
n1   = spark.read.parquet(out).count()
max1 = spark.read.parquet(out).agg({"value": "max"}).collect()[0][0]
print(f"After run 1: {n1} rows; max value = {max1}")

# Run 2 — same checkpoint, so Spark resumes from where run 1 stopped.
run_for(6)
n2   = spark.read.parquet(out).count()
max2 = spark.read.parquet(out).agg({"value": "max"}).collect()[0][0]
print(f"After run 2: {n2} rows; max value = {max2}")
print(f"Run 2 added {n2 - n1} new rows; the max value strictly increased — proof of resume.")

# Peek inside the checkpoint directory.
print("\nCheckpoint contents:")
for entry in sorted(Path(ckpt).iterdir()):
    print(" ", entry.name)

# 08 · Performance Tuning

## Q25 — Reduce 1000 partitions to 100 without a shuffle

**Scenario.** Output side has 1000 small partitions; you want roughly 100 output files.

**Answer.** `df.coalesce(100)`.

**Why.** `coalesce(N)` merges partitions *locally* — no shuffle. Use it when you are *reducing* the count.

`repartition(N)` does a full shuffle. Use it when **increasing** the partition count or when you need to rebalance skewed partitions.

**Trap.** `repartition(100)` on 1000 partitions — correct result, but pays a full shuffle for no reason.

## Q26 — Joining a tiny dimension against a huge fact: stop the shuffle

**Scenario.** 5 TB fact joined against a 30 MB dimension; the job spends 80% of its time shuffling.

**Answer.** **Broadcast the smaller side**.

```python
 from pyspark.sql.functions import broadcast
 fact.join(broadcast(dim), on="merchant_id", how="left")
```

**Why.** Broadcasting ships the small side to every executor, eliminating the shuffle of the large side. Spark also **auto-broadcasts** when the small side's estimated size is below `spark.sql.autoBroadcastJoinThreshold` (default 10 MB; bump to 100 MB+ in production).

**Trap.** Broadcasting a side that doesn't actually fit in driver memory — that blows the driver.

## Q27 — Driver GC logs full of pauses: what's the real cause?

**Scenario.** Driver GC time climbs; the notebook is slow / hangs.

**Answer.** The driver is being asked to hold too much data. Three real fixes, in order:

1. **Stop collecting big results to the driver** — `collect()`, `toPandas()`, and `take(N)` with huge N all pull rows into the driver JVM. Write to a table instead.
2. **Lower `spark.sql.autoBroadcastJoinThreshold`** — broadcasting a side larger than the driver can hold also blows it up.
3. **As a last resort, bump `spark.driver.memory`** and possibly tune the driver GC.

**Why the distinction matters.** Executor GC pressure is a different problem with a different fix — repartition / increase executor memory. Driver GC pressure almost always means *too much is being collected to the driver*, and bumping memory just delays the inevitable.

**Trap.** Bumping `spark.executor.memory` for a driver-side problem.

## Q28 — Which of these operations causes a shuffle?

**Scenario.** Pick the shuffle-causing (wide) operations from this list:

```python
 df.filter(col("amount") > 100)              # ?
 df.select("customer_id", "amount")          # ?
 df.withColumn("x", lit(1))                  # ?
 df.union(other)                             # ?
 df.coalesce(8)                              # ?
 df.groupBy("customer_id").sum("amount")     # ?
 df.join(other, "customer_id")               # ? (non-broadcast)
 df.distinct()                               # ?
 df.repartition(8)                           # ?
 df.orderBy("amount")                        # ?
```

**Answer.** **Wide (shuffle):** `groupBy + agg`, non-broadcast `join`, `distinct` / `dropDuplicates`, `repartition`, `orderBy` / `sort`, window functions, `intersect`, `except`. **Narrow (no shuffle):** `select`, `filter`, `withColumn`, `union`, `coalesce`, `mapPartitions`.

**Why.** A **wide** transformation needs rows from *multiple* input partitions to produce a single output partition, which requires moving data across the network — a **shuffle**. A **narrow** transformation processes each input partition independently. Spark draws a **stage boundary at every shuffle**, so wide operations also create new stages and write intermediate shuffle files.

**The RDD analogs the exam may show:**

| RDD operation | Wide? | DataFrame analog |
|---|---|---|
| `groupByKey` | ✅ Yes | `groupBy + agg` |
| `reduceByKey` | ✅ Yes (with map-side combine) | `groupBy + agg` |
| `aggregateByKey` | ✅ Yes | `groupBy + agg` |
| `join` | ✅ Yes | `join` (non-broadcast) |
| `distinct` | ✅ Yes | `distinct` |
| `sortByKey` | ✅ Yes | `orderBy` |
| `repartition` | ✅ Yes | `repartition` |
| `map` | ❌ No | `select` / `withColumn` |
| `filter` | ❌ No | `filter` / `where` |
| `flatMap` | ❌ No | `select` + `explode` |
| `mapPartitions` | ❌ No | `mapPartitions` |
| `union` | ❌ No | `union` |
| `coalesce` (decrease) | ❌ No | `coalesce` |

**Traps:**

- **`reduceByKey` vs. `groupByKey`** — both shuffle, but `reduceByKey` combines values **map-side first**, so much less data crosses the network. Don't say `reduceByKey` is narrow — it isn't — but do prefer it over `groupByKey` when you're aggregating.
- **`coalesce` is narrow only when reducing partition count** — Spark merges partitions locally. To *increase* partitions you need `repartition`, which shuffles.
- **`union`** is narrow — it just concatenates partitions, no shuffle. Don't confuse it with SQL `UNION` (which deduplicates and would need a shuffle); Spark's `union` is closer to `UNION ALL`.
- **`join` becomes narrow** when one side is broadcast (no shuffle of the big side).

In [ ]:
# Read the physical plans — "Exchange" is the shuffle marker.
from pyspark.sql.functions import col, sum as _sum

print("=" * 60, "\nfilter — NARROW (no Exchange in the plan)\n", "=" * 60, sep="")
df.filter(col("credit_score") > 700).explain()

print("=" * 60, "\ngroupBy + sum — WIDE (look for 'Exchange hashpartitioning')\n", "=" * 60, sep="")
df.groupBy("city").agg(_sum("credit_score")).explain()

print("=" * 60, "\norderBy — WIDE (look for 'Exchange rangepartitioning')\n", "=" * 60, sep="")
df.orderBy("credit_score").explain()

print("=" * 60, "\nrepartition — WIDE (Exchange RoundRobinPartitioning)\n", "=" * 60, sep="")
df.repartition(4).explain()

print("=" * 60, "\ncoalesce — NARROW (no Exchange — partitions merged locally)\n", "=" * 60, sep="")
df.coalesce(2).explain()

# 09 · Pandas API on Spark

## Q29 — Pandas-on-Spark DataFrame → Spark DataFrame

**Scenario.** You wrote a feature-engineering step in pandas-on-Spark (`pyspark.pandas`) and need to hand it off to code that expects a regular Spark DataFrame.

**Answer.** `.to_spark()`.

```python
 import pyspark.pandas as ps
 psdf = ps.DataFrame({"customer_id": ["C1","C2"], "score": [700, 720]})
 sdf  = psdf.to_spark()        # → pyspark.sql.DataFrame
```

**The reverse direction:**

- From a Spark DataFrame to pandas-on-Spark: `sdf.pandas_api()` (or, on older versions, `sdf.to_pandas_on_spark()`).
- From a Spark DataFrame to *real* pandas (collects to the driver — careful!): `sdf.toPandas()`.

**Trap.** Calling `.toPandas()` on a big Spark DataFrame in any pipeline. It collects to the driver and OOMs.

In [ ]:
import pyspark.pandas as ps

psdf = ps.DataFrame({"customer_id": ["C1", "C2"], "score": [700, 720]})
sdf  = psdf.to_spark()

print("type(sdf) =", type(sdf).__name__)
sdf.show()

## Summary

29 sample questions, grouped to mirror the notebooks they live next to. Four patterns to internalise:

- **Wide vs. narrow transformations** (Q28) — `groupBy`, `join`, `distinct`, `repartition`, `orderBy`, window functions shuffle. `select`, `filter`, `withColumn`, `union`, `coalesce` don't. The Spark UI confirms: every shuffle is a new stage with an `Exchange` operator in the plan.
- **Driver GC vs. executor GC** (Q27) — driver-side almost always means you're collecting too much; bumping driver memory is the *last* lever, not the first.
- **`recursiveFileLookup` and partition discovery** (Q7) — they don't compose. Pick one: deep arbitrary walk *or* `year=…/month=…` partition columns. The combo answer is `basePath` + Hive-style partitions, without `recursiveFileLookup`.
- **`checkpointLocation` is part of the data, not the code** (Q24) — delete it and the next run starts from offset 0. One dedicated path per streaming query, on durable cloud storage.

If you can answer all 29 cold, you're past the foundations-level questions on the exam.

In [ ]:
spark.stop()